# Multi-Head Attention
### Why settle for one perspective when you can have eight?

---

## 📖 The Story

It is still 2017. Self-attention (Topic 2) just replaced the RNN entirely — but the team behind "Attention Is All You Need" notices a limitation. A single attention head produces only ONE weighted average per token. One Query, one Key, one Value, one attention distribution per word.

But language has many relationship types layered at once — syntax, coreference, position — and one head has to blend all of them into a single softmax row. So they ask: *"What if we ran several smaller attention operations in parallel, each free to specialize, then combined their findings?"*

That is multi-head attention. This notebook builds it from scratch: split, attend, concatenate, project.

---

**What this notebook covers:**
- Splitting Q, K, V into multiple heads
- Running scaled dot-product attention independently inside each head
- Concatenating head outputs and applying the final output projection Wᴼ
- Visualizing how different heads can learn to focus on different relationships
- Verifying our implementation against PyTorch's `nn.MultiheadAttention`

**Prerequisites:**
- Why Attention Matters (Topic 1)
- Self-Attention (Topic 2)
- Matrix reshaping and concatenation

**Note:** Like Topics 1 and 2, this notebook uses an in-notebook example sentence rather than an external dataset — multi-head attention is best understood through direct, inspectable computation on a small example.

In [ ]:
# --- All Imports ---
import numpy as np                        # Core math — multi-head attention from scratch
import matplotlib.pyplot as plt           # Plotting
import seaborn as sns                     # Heatmap visualizations
import torch                              # PyTorch — verification
import torch.nn as nn                     # nn.MultiheadAttention
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)                        # Reproducibility
torch.manual_seed(42)

print("All imports successful ✅")

## Part 1: Theory Recap

Five things to know before we write any code:

- **One head = one averaged view**: A single self-attention head (Topic 2) produces exactly one attention distribution per token — it must blend every type of relationship (syntax, coreference, position) into that one view.
- **Multiple heads = multiple specialized views**: Multi-head attention runs several smaller self-attention operations in parallel, each on its own learned slice of the embedding space, so different heads can specialize in different relationship types.
- **The split keeps cost comparable**: If d_model is split across h heads, each head works with dₖ = d_model / h — so running h heads costs roughly the same as one full-size head, not h times more.
- **The formula per head is unchanged from Topic 2**: headᵢ = softmax((QWᵢQ)(KWᵢK)ᵀ/√dₖ)(VWᵢV) — this is exactly scaled dot-product attention, just applied per head.
- **Concat + Wᴼ recombines the heads**: After running all heads independently, their outputs are concatenated back into the original d_model size, then passed through a learned output projection Wᴼ that mixes information across heads.

## Setting Up a Real Example Sentence

We use the sentence **"The cat sat on mat"** — the same sentence from Topics 1 and 2 — so you can directly compare single-head self-attention (Topic 2) with multi-head self-attention (this notebook).

Each word gets a simple embedding vector. In a real transformer these come from a learned embedding layer; here we use small random vectors so the math stays inspectable.

In [ ]:
# Define our example sentence
words = ["The", "cat", "sat", "on", "mat"]
seq_len   = len(words)
d_model   = 8   # total embedding dimension
num_heads = 2   # number of attention heads
d_k       = d_model // num_heads   # dimension PER head — this is the key split

assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

# Create input embeddings — one vector per word
# In a real model these come from a trained embedding layer
X = np.random.randn(seq_len, d_model) * 0.5

print("Input embeddings X:")
print(f"Shape: {X.shape}  (seq_len={seq_len}, d_model={d_model})")
print(f"Splitting into {num_heads} heads → each head gets d_k = {d_k}\n")
for word, vec in zip(words, X):
    print(f"  {word:6s}: {np.round(vec, 2)}")

## Part 2: Multi-Head Attention From Scratch

We now build multi-head attention step by step:
1. Create PER-HEAD learned weight matrices WᵢQ, WᵢK, WᵢV (one set per head)
2. Project X into Qᵢ, Kᵢ, Vᵢ for each head independently
3. Run scaled dot-product attention (Topic 2's formula) inside each head
4. Concatenate all head outputs back into one (seq_len, d_model) matrix
5. Apply the final output projection Wᴼ

In [ ]:
def softmax(x, axis=-1):
    """
    Numerically stable softmax along a given axis.
    Subtract row max before exp to prevent overflow.
    """
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)


# Step 1: Create learned weight matrices for EACH head
# In a real model these are learned via backpropagation.
# Here we initialize them randomly to simulate a trained state.
# INTERVIEW NOTE: each head gets its OWN W_Q, W_K, W_V — they do not share weights.
W_Q_heads = [np.random.randn(d_model, d_k) * 0.5 for _ in range(num_heads)]
W_K_heads = [np.random.randn(d_model, d_k) * 0.5 for _ in range(num_heads)]
W_V_heads = [np.random.randn(d_model, d_k) * 0.5 for _ in range(num_heads)]

# Final output projection — shared across heads, applied AFTER concatenation
W_O = np.random.randn(num_heads * d_k, d_model) * 0.5

print(f"Number of heads: {num_heads}")
print(f"Per-head W_Q shape: {W_Q_heads[0].shape}  (one of {num_heads} independent sets)")
print(f"Output projection W_O shape: {W_O.shape}")

In [ ]:
def multi_head_attention(X, W_Q_heads, W_K_heads, W_V_heads, W_O, d_k, num_heads):
    """
    Multi-Head Attention.
    MultiHead(Q,K,V) = Concat(head_1, ..., head_h) W_O
    where head_i = softmax((X W_Qi)(X W_Ki)^T / sqrt(d_k)) (X W_Vi)

    Args:
        X            : input embeddings — shape (seq_len, d_model)
        W_Q_heads    : list of per-head Query weight matrices
        W_K_heads    : list of per-head Key weight matrices
        W_V_heads    : list of per-head Value weight matrices
        W_O          : final output projection matrix
        d_k          : dimension per head (for scaling)
        num_heads    : number of heads

    Returns:
        output       : final multi-head output — shape (seq_len, d_model)
        head_outputs : list of per-head outputs — shape (seq_len, d_k) each
        head_weights : list of per-head attention weight matrices — shape (seq_len, seq_len) each
    """
    head_outputs = []
    head_weights = []

    for i in range(num_heads):
        # Step 2: Project X into this head's own Q, K, V
        Q_i = X @ W_Q_heads[i]   # shape: (seq_len, d_k)
        K_i = X @ W_K_heads[i]
        V_i = X @ W_V_heads[i]

        # Step 3: Scaled dot-product attention — IDENTICAL formula to Topic 2, just per head
        scores_i = (Q_i @ K_i.T) / np.sqrt(d_k)
        weights_i = softmax(scores_i, axis=-1)
        out_i = weights_i @ V_i   # shape: (seq_len, d_k)

        head_outputs.append(out_i)
        head_weights.append(weights_i)
        print(f"  Head {i+1}: Q,K,V shape {Q_i.shape} -> attention weights {weights_i.shape} -> output {out_i.shape}")

    # Step 4: Concatenate all head outputs along the feature dimension
    concat = np.concatenate(head_outputs, axis=-1)   # shape: (seq_len, num_heads * d_k) = (seq_len, d_model)
    print(f"\n  Concatenated heads shape: {concat.shape}  (back to d_model = {num_heads} x {d_k})")

    # Step 5: Final output projection
    output = concat @ W_O   # shape: (seq_len, d_model)
    print(f"  After W_O projection: {output.shape}  (same shape as input — context-aware now)")

    return output, head_outputs, head_weights


print("Running multi-head attention on 'The cat sat on mat':\n")
mha_output, head_outputs, head_weights = multi_head_attention(
    X, W_Q_heads, W_K_heads, W_V_heads, W_O, d_k, num_heads
)

## What Just Happened?

We just computed multi-head attention for an entire sentence using two independent attention heads.

- Each head projected the SAME input X into its own Q, K, V using its own private weight matrices
- Each head ran the exact scaled dot-product attention formula from Topic 2, completely independently
- Head 1 and Head 2 produced DIFFERENT attention weight matrices over the same 5 words — because their Q/K/V projections differ
- We concatenated both heads' outputs side by side, restoring the original d_model dimension
- The final Wᴼ projection blended information across both heads into one unified representation

Notice that no single step here is new mathematically — multi-head attention is simply Topic 2's formula, applied several times in parallel, then recombined.

In [ ]:
# --- Visualisation 1: Comparing Attention Heatmaps Across Heads ---
# Unlike Topic 2's single heatmap, we now have ONE heatmap PER HEAD —
# this is exactly what lets us inspect whether heads specialize differently.

fig, axes = plt.subplots(1, num_heads, figsize=(7 * num_heads, 6))
if num_heads == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    sns.heatmap(
        head_weights[i],
        xticklabels=words,
        yticklabels=words,
        annot=True,
        fmt='.2f',
        cmap='YlOrRd',
        linewidths=0.5,
        ax=ax,
        cbar_kws={'label': 'Attention Weight'}
    )
    ax.set_title(f'Head {i+1} Attention Weights', fontsize=13, fontweight='bold')
    ax.set_xlabel('Key (being attended TO)', fontsize=11)
    ax.set_ylabel('Query (doing the attending)', fontsize=11)

plt.suptitle('Multi-Head Attention — Each Head Learns a Different View of "The cat sat on mat"',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Even with random (untrained) weights, notice the two heads already produce")
print("DIFFERENT attention patterns over the SAME sentence — because each head has")
print("its own independent W_Q, W_K, W_V. After real training, heads like these tend")
print("to specialize into interpretable roles: syntax, coreference, position, etc.")

In [ ]:
# --- Visualisation 2: How Concatenation Restores d_model ---
# A simple bar chart showing how each head contributes a SLICE of the
# final d_model-dimensional output, and how W_O blends them together.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: per-head output dimension vs combined dimension
dims = [d_k] * num_heads + [d_model]
labels = [f'Head {i+1}\noutput' for i in range(num_heads)] + ['Concat\n(restored\nd_model)']
colors = sns.color_palette('Set2', num_heads) + [('#444444',)]
bar_colors = list(sns.color_palette('Set2', num_heads)) + ['#444444']

axes[0].bar(labels, dims, color=bar_colors)
axes[0].set_title(f'Per-Head Dimension (d_k={d_k}) vs Concatenated Dimension (d_model={d_model})',
                  fontsize=11, fontweight='bold')
axes[0].set_ylabel('Vector Dimension', fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# Right: average attention weight magnitude per head, to show heads ARE behaving differently
avg_weights_per_head = [w.mean(axis=0) for w in head_weights]  # average row per head
x_pos = np.arange(len(words))
width = 0.8 / num_heads
for i, avg_w in enumerate(avg_weights_per_head):
    axes[1].bar(x_pos + i * width, avg_w, width=width, label=f'Head {i+1}', color=bar_colors[i])

axes[1].set_xticks(x_pos + width * (num_heads - 1) / 2)
axes[1].set_xticklabels(words)
axes[1].set_title('Average Attention Received by Each Word, Per Head', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Average Attention Weight', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Multi-Head Attention: Dimension Split and Per-Head Behavior', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Each head outputs a {d_k}-dimensional vector per token.")
print(f"Concatenating {num_heads} heads of size {d_k} restores the original d_model={d_model}.")
print("The right plot shows each head distributes its average attention differently across")
print("the sentence — this is the raw material W_O learns to combine.")

## Part 3: PyTorch Verification

We verify our scratch implementation using PyTorch's official `nn.MultiheadAttention` module — the exact layer used inside real transformer implementations, including the original Transformer, BERT, and GPT.

To compare fairly, we manually load OUR weight matrices into PyTorch's module, so both implementations run on identical weights.

In [ ]:
# Build a PyTorch nn.MultiheadAttention layer with matching dimensions
torch_mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, bias=False, batch_first=True)

# PyTorch stores all heads' Q, K, V weights concatenated together internally.
# We reconstruct that combined weight matrix from OUR per-head weights so both
# implementations are mathematically equivalent before comparing outputs.
W_Q_combined = np.concatenate(W_Q_heads, axis=1)   # shape: (d_model, d_model)
W_K_combined = np.concatenate(W_K_heads, axis=1)
W_V_combined = np.concatenate(W_V_heads, axis=1)

with torch.no_grad():
    # PyTorch expects in_proj_weight as a single (3*d_model, d_model) matrix: [Q; K; V] stacked,
    # and applies it as X @ W.T — so we transpose our combined weights to match.
    torch_mha.in_proj_weight.copy_(
        torch.tensor(np.concatenate([W_Q_combined.T, W_K_combined.T, W_V_combined.T], axis=0), dtype=torch.float32)
    )
    torch_mha.out_proj.weight.copy_(torch.tensor(W_O.T, dtype=torch.float32))

X_torch = torch.tensor(X, dtype=torch.float32).unsqueeze(0)  # (1, seq_len, d_model)

with torch.no_grad():
    output_torch, weights_torch = torch_mha(X_torch, X_torch, X_torch, average_attn_weights=False)

output_torch_np = output_torch.squeeze(0).numpy()  # (seq_len, d_model)

print("=== Scratch vs PyTorch Verification ===\n")
print("Output (first row — representation of 'The'):")
print(f"  Scratch : {np.round(mha_output[0], 5)}")
print(f"  PyTorch : {np.round(output_torch_np[0], 5)}")

max_diff = np.max(np.abs(mha_output - output_torch_np))
print(f"\nMax absolute difference across full output: {max_diff:.2e}")
print("✅ Match — our scratch implementation matches PyTorch's official module" if max_diff < 1e-3
      else "❌ Mismatch — check weight reshaping or transposition")

## Part 4: Hyperparameter Experiments

Two experiments to build deeper intuition:

1. **Effect of number of heads on per-head dimension** — as h grows (with d_model fixed), how does dₖ shrink, and what does that mean for each head's expressiveness?
2. **Computational cost: multi-head vs an equivalent single large head** — directly comparing total FLOPs to confirm multi-head attention does not add overhead beyond single-head attention.

In [ ]:
# Experiment 1: Vary number of heads (with d_model fixed) and observe d_k shrinking
d_model_fixed = 64
head_counts = [1, 2, 4, 8, 16, 32]
d_k_per_count = [d_model_fixed // h for h in head_counts]

# Experiment 2: Total QK^T FLOPs — multi-head (h heads of size d_k) vs single head (1 head of size d_model)
# Single head:   seq_len^2 * d_model
# Multi-head:    h * (seq_len^2 * d_k)  =  h * (seq_len^2 * d_model/h)  =  seq_len^2 * d_model  (same!)
seq_len_demo = 10
single_head_flops = [seq_len_demo**2 * d_model_fixed for _ in head_counts]
multi_head_flops  = [h * (seq_len_demo**2 * (d_model_fixed // h)) for h in head_counts]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: d_k shrinking as heads increase
axes[0].plot(head_counts, d_k_per_count, marker='o', color='steelblue', linewidth=2, markersize=8)
axes[0].set_title(f'Per-Head Dimension Shrinks as Heads Increase\n(d_model fixed at {d_model_fixed})',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Heads (h)', fontsize=11)
axes[0].set_ylabel('Per-Head Dimension (d_k = d_model / h)', fontsize=11)
axes[0].set_xscale('log', base=2)
axes[0].grid(True, alpha=0.3)
for x, y in zip(head_counts, d_k_per_count):
    axes[0].annotate(f'{y}', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)

# Plot 2: total FLOPs stay constant regardless of head count
axes[1].plot(head_counts, single_head_flops, marker='s', color='red', linewidth=2,
             markersize=8, label='Equivalent single large head')
axes[1].plot(head_counts, multi_head_flops, marker='o', color='green', linewidth=2,
             markersize=6, linestyle='--', label='Multi-head (sum across heads)')
axes[1].set_title('Total QKᵀ FLOPs: Multi-Head vs Single-Head\n(Same total cost!)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Number of Heads (h)', fontsize=11)
axes[1].set_ylabel(f'FLOPs (seq_len={seq_len_demo})', fontsize=11)
axes[1].set_xscale('log', base=2)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Multi-Head Attention Experiments: Dimension Split & Computational Cost',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"As heads increase from {head_counts[0]} to {head_counts[-1]}, each head's dimension shrinks from")
print(f"{d_k_per_count[0]} to {d_k_per_count[-1]} — but total QK^T FLOPs stay IDENTICAL at every head count.")
print("This confirms: splitting into more heads adds representational diversity for free,")
print("without increasing the theoretical computational cost.")

## Common Mistakes Students Make

**Mistake 1: Thinking each head sees a different INPUT**
Every head sees the exact SAME input embeddings X. What differs between heads is only the learned projection matrices WᵢQ, WᵢK, WᵢV — each head learns to extract a different kind of information from the same data, not from different data.

**Mistake 2: Forgetting to scale by √dₖ (per-head), not √d_model**
The scaling factor inside each head's attention must use that head's OWN dimension dₖ = d_model / h, not the full d_model. Using the wrong scaling factor breaks the variance-stabilizing property explained in Topic 2.

**Mistake 3: Skipping the final output projection Wᴼ**
Concatenating head outputs alone is not enough — without Wᴼ, the model has no learned way to combine information ACROSS heads. Wᴼ is what allows, for example, a coreference signal from Head 1 to interact with a syntax signal from Head 2 in the final representation.

## Part 5: Interview Corner

**Question: Walk me through why multi-head attention helps with token interactions more than a single head does.**

This directly extends the module's learning outcome about token interactions through attention. Complete answer:

**Setup:** A single self-attention head computes ONE attention distribution per token — a single weighted view of how every token relates to every other token.

**Step 1 — The limitation:** Real language relationships are not one-dimensional. The same token might need to attend strongly to one neighbor for grammatical agreement and to a completely different token for semantic meaning. One shared distribution forces a compromise between these needs.

**Step 2 — The fix:** Multi-head attention creates h independent copies of the attention mechanism, each with its own WᵢQ, WᵢK, WᵢV. Because the projections differ, each head is free to learn a DIFFERENT notion of "relevance" between tokens.

**Step 3 — Specialization:** During training, heads naturally diverge — empirically, researchers have found heads that track adjacent-token position, heads that track subject-verb syntax, and heads that resolve coreference (like "it" → "animal"), all within the same layer.

**Step 4 — Recombination:** Concat + Wᴼ takes all of these specialized "opinions" and lets the model learn how to weigh and combine them into one final token representation — richer than any single head could produce alone.

**The key insight:** Multi-head attention does not change the fundamental Query-Key-Value interaction from Topic 2 — it multiplies it, in parallel, across independently learned subspaces, then gives the model a learned way (Wᴼ) to merge those independent views back together. This is exactly why a single transformer layer can simultaneously track grammar, meaning, and position in one forward pass.

## Key Takeaways

Five bullets you must remember for placement interviews:

- **Multi-head attention runs several self-attention operations in parallel**: Each head has its own learned WᵢQ, WᵢK, WᵢV and operates on the SAME input, but extracts a different kind of relationship.
- **The per-head formula is unchanged from Topic 2**: headᵢ = softmax((QWᵢQ)(KWᵢK)ᵀ/√dₖ)(VWᵢV) — multi-head attention is a structural change, not a new mathematical operation.
- **Splitting d_model across heads keeps cost comparable**: With dₖ = d_model / h, running h heads costs about the same in total FLOPs as one full-size head — diversity comes essentially for free.
- **Concat + Wᴼ recombines specialized views into one representation**: Concatenation alone just stacks head outputs side by side; the learned output projection Wᴼ is what lets the model blend information ACROSS heads.
- **Different heads tend to specialize**: Empirically, heads learn to track position, syntax, or coreference — distinct, often interpretable behaviors that emerge purely from training, without being explicitly assigned.